# Solver Runtime Scaling — Fig. 4

Paper-facing skeleton that produces Fig. 4 (log-log wall-clock vs. resolution, three A* backends) for the NeurIPS 2026 submission. The developer walkthrough at `../../notebooks/solve.ipynb` remains the long-form exploration notebook; this one is the paper-facing export and is intentionally minimal.

Source of truth for paper structure: `../../../passage-paper/neurips/skeleton.md` (v3).

## Paper Mapping

| Claim | `\evidence{}` tag | Output artifact |
| --- | --- | --- |
| Sec. 5.1 solver backends scale differently with resolution | `passage/paper/notebooks/solver_runtime_scaling.ipynb -> passage/paper/figures/solver_runtime_scaling.pdf` | `figures/solver_runtime_scaling.pdf` |
| Numba-JIT A* is the only backend feasible at 4096x4096 | `passage/paper/notebooks/solver_runtime_scaling.ipynb -> solver_runtime_scaling.csv` | `figures/solver_runtime_scaling.csv` |

## Inputs / Outputs

| Kind | Path |
| --- | --- |
| Input | One representative tile at each resolution in `{64, 128, 256, 512, 1024, 2048, 4096}` |
| Input | Solver backends `{networkx, pathfinding, passage_grid}` |
| Input | Heuristics `{euclidean, manhattan, chebyshev}` |
| Output | `../figures/solver_runtime_scaling.pdf` (+ `.png`) |
| Output | `../figures/solver_runtime_scaling.csv` |
| Output | `../figures/solver_runtime_scaling.manifest.json` |


In [ ]:
# Setup: import helpers and handle optional dependencies.
from __future__ import annotations

import sys
from pathlib import Path

PAPER_DIR = Path.cwd().resolve().parent
if str(PAPER_DIR) not in sys.path:
    sys.path.insert(0, str(PAPER_DIR))

try:
    from helpers import FIGURES_DIR, emit_manifest, run_astar_bench, save_figure
except ImportError as exc:  # pragma: no cover
    raise ImportError(
        "Could not import passage.paper.helpers; run from passage/paper/notebooks/."
    ) from exc

try:
    import numpy as np
    import matplotlib.pyplot as plt
    HAS_PLOTTING = True
except ImportError:
    HAS_PLOTTING = False

try:
    import passage  # noqa: F401
    HAS_PASSAGE = True
except ImportError:
    HAS_PASSAGE = False

STUDY_ID = "solver_runtime_scaling"
RESOLUTIONS = [64, 128, 256, 512, 1024, 2048, 4096]
BACKENDS = ["networkx", "pathfinding", "passage_grid"]
HEURISTIC = "euclidean"
COST_MODEL = "distance"
WEIGHT = 0.0
TILE_SEED = 42

In [ ]:
# Run the timed sweep. Each cell of the sweep is one (backend, resolution)
# pair; when the real data share is not mounted the helper short-circuits
# the run and marks the row status='skipped_*'.
#
# TODO: replace the synthetic grid with real loaded tiles and extend the
# sweep with heuristics and cost models as needed for the final figure.
# The developer walkthrough at ../../notebooks/solve.ipynb carries the
# long-form parameter sweep.

rows = []
if HAS_PLOTTING:
    rng = np.random.default_rng(TILE_SEED)
    for resolution in RESOLUTIONS:
        grid = rng.random((resolution, resolution)).astype("float32")
        start = (0, 0)
        goal = (resolution - 1, resolution - 1)
        for backend in BACKENDS:
            result = run_astar_bench(
                backend=backend,
                grid=grid,
                start=start,
                goal=goal,
                cost_model=COST_MODEL,
                weight=WEIGHT,
                heuristic=HEURISTIC,
            )
            rows.append(result.to_dict())

csv_path = FIGURES_DIR / f"{STUDY_ID}.csv"
if rows:
    header = list(rows[0].keys())
    lines = [",".join(header)]
    for row in rows:
        lines.append(",".join(str(row[k]) for k in header))
    csv_path.write_text("\n".join(lines) + "\n")
else:
    csv_path.write_text("backend,resolution,wall_clock_s,status\n")
print(f"wrote {csv_path}")

In [ ]:
# Build the log-log scaling figure (Fig. 4).
if HAS_PLOTTING and rows:
    fig, ax = plt.subplots(figsize=(5.5, 4.0))
    for backend in BACKENDS:
        xs = [r["resolution"] for r in rows if r["backend"] == backend and r["wall_clock_s"] == r["wall_clock_s"]]  # drop NaN
        ys = [r["wall_clock_s"] for r in rows if r["backend"] == backend and r["wall_clock_s"] == r["wall_clock_s"]]
        if xs and ys:
            ax.loglog(xs, ys, marker="o", label=backend)
    ax.set_xlabel("grid resolution (pixels per side)")
    ax.set_ylabel("wall-clock per sample (s)")
    ax.set_title("A* runtime vs. resolution (stub data — replace with real tiles)")
    ax.grid(True, which="both", linestyle=":", alpha=0.5)
    ax.legend()
    fig.tight_layout()
else:
    fig = None

In [ ]:
# Emit manifest and save figure.
outputs = [csv_path]
if fig is not None:
    pdf_path = save_figure(fig, study_id=STUDY_ID)
    outputs.extend([pdf_path, pdf_path.with_suffix(".png")])

manifest_path = emit_manifest(
    study_id=STUDY_ID,
    inputs={
        "resolutions": RESOLUTIONS,
        "backends": BACKENDS,
        "heuristic": HEURISTIC,
        "cost_model": COST_MODEL,
        "weight": WEIGHT,
        "tile_seed": TILE_SEED,
        "has_plotting": HAS_PLOTTING,
        "has_passage_package": HAS_PASSAGE,
        "paper_section": "5.1",
        "paper_anchors": ["fig:solver-runtime"],
    },
    outputs=outputs,
)
print(f"wrote {manifest_path}")